In [1]:
import rootutils

root_dir = rootutils.setup_root(".",
                                indicator=".project-root",
                                pythonpath=True)
from src.main import main
from src.cache import clear_entire_cache, clear_cache_for_molecule
import json
from src.utils.az import is_basic_molecule
from src.protecting_group import identify_protection_sites, suggest_protecting_groups, get_protection_recommendations, format_recommendations_for_prompt, ProtectionSelection, create_protection_selection, create_protection_plan, hitl_workflow_example, format_sites_for_display, format_pg_options_for_display, canonicalize_and_map_atoms, hitl_workflow_example

In [2]:
smiles = "CCO[C@]1(C)C[C@](C)C(=O)[C@@](C)[C@](OCC)[C@](C)(OCC)[C@](CC)OC(=O)[C@@](C)[C@](O[C@]2C[C@@](C)(OC)[C@](O)[C@@](C)O2)[C@@](C)[C@]1O[C@]1O[C@](C)C[C@](N(C)C)[C@@]1OCC"


In [3]:
mapped, amap, mol = canonicalize_and_map_atoms(smiles)
mapped

'[CH3:1][CH2:2][O:3][C@H:4]1[C@H:5]([O:6][C@H:7]2[C@H:8]([CH3:9])[C@@H:10]([O:11][C@@H:12]3[CH2:13][C@@:14]([CH3:15])([O:16][CH3:17])[C@H:18]([OH:19])[C@@H:20]([CH3:21])[O:22]3)[C@H:23]([CH3:24])[C:25](=[O:26])[O:27][C@@H:28]([CH2:29][CH3:30])[C@@:31]([CH3:32])([O:33][CH2:34][CH3:35])[C@@H:36]([O:37][CH2:38][CH3:39])[C@H:40]([CH3:41])[C:42](=[O:43])[C@@H:44]([CH3:45])[CH2:46][C@@:47]2([CH3:48])[O:49][CH2:50][CH3:51])[O:52][C@H:53]([CH3:54])[CH2:55][C@@H:56]1[N:57]([CH3:58])[CH3:59]'

In [4]:
identify_protection_sites(smiles)

[ProtectionSite(site_id='9777149c694d', atom_map_numbers=(18, 14, 20, 19), functional_group_id='secondary_alcohol', functional_group_name='Secondary Alcohol', category='alcohol', reactivity='medium', compatible_pgs=['TBS', 'TBDPS', 'Bn', 'Ac', 'PMB', 'THP', 'TES']),
 ProtectionSite(site_id='e813b272ea28', atom_map_numbers=(57, 56, 58, 59), functional_group_id='tertiary_amine', functional_group_name='Tertiary Amine', category='amine', reactivity='low', compatible_pgs=[]),
 ProtectionSite(site_id='fce3c9dcd35c', atom_map_numbers=(42, 43, 40, 44), functional_group_id='ketone', functional_group_name='Ketone', category='carbonyl', reactivity='high', compatible_pgs=['ketal', 'dithiane', 'dithiolane']),
 ProtectionSite(site_id='b749e2d74662', atom_map_numbers=(25, 26, 27, 28), functional_group_id='ester', functional_group_name='Ester', category='ester', reactivity='low', compatible_pgs=[])]

In [5]:
recs = get_protection_recommendations(smiles, mode="hitl")
for rec in recs:
    print(f"{rec.site.functional_group_name}: {rec.suggestions}")

Secondary Alcohol: [PGSuggestion(protecting_group_id='TBS', name='tert-Butyldimethylsilyl', abbreviation='TBS', score=0.8049999999999999, protection_reagent='TBSCl, imidazole, DMF or TBSOTf, 2,6-lutidine', deprotection_conditions=['TBAF', 'HF-pyridine', 'AcOH', 'PPTS/MeOH'], stability={'acid': 'low', 'base': 'high', 'hydrogenation': 'high', 'oxidation': 'high'}, orthogonality_score=0.85), PGSuggestion(protecting_group_id='Bn', name='Benzyl', abbreviation='Bn', score=0.7999999999999999, protection_reagent='BnBr, NaH, DMF or BnBr, Ag2O', deprotection_conditions=['H2/Pd-C', 'Na/NH3 (Birch)', 'DDQ'], stability={'acid': 'high', 'base': 'high', 'hydrogenation': 'low', 'oxidation': 'medium'}, orthogonality_score=0.8), PGSuggestion(protecting_group_id='TBDPS', name='tert-Butyldiphenylsilyl', abbreviation='TBDPS', score=0.7849999999999999, protection_reagent='TBDPSCl, imidazole, DMF', deprotection_conditions=['TBAF', 'HF-pyridine'], stability={'acid': 'medium', 'base': 'high', 'hydrogenation': 

In [6]:
print(
    hitl_workflow_example(
        "CCO[C@]1(C)C[C@](C)C(=O)[C@@](C)[C@](OCC)[C@](C)(OCC)[C@](CC)OC(=O)[C@@](C)[C@](O[C@]2C[C@@](C)(OC)[C@](O)[C@@](C)O2)[C@@](C)[C@]1O[C@]1O[C@](C)C[C@](N(C)C)[C@@]1OCC"
    ))

HITL Workflow Example

Molecule: CCO[C@]1(C)C[C@](C)C(=O)[C@@](C)[C@](OCC)[C@](C)(OCC)[C@](CC)OC(=O)[C@@](C)[C@](O[C@]2C[C@@](C)(OC)[C@](O)[C@@](C)O2)[C@@](C)[C@]1O[C@]1O[C@](C)C[C@](N(C)C)[C@@]1OCC

STEP 1: All Identified Protection Sites (5 found)
--------------------------------------------------
  [0] Secondary Alcohol (alcohol) - medium reactivity - atoms (18, 14, 20, 19) - PGs available: yes
  [1] Tertiary Amine (amine) - low reactivity - atoms (57, 56, 58, 59) - PGs available: no
  [2] Ketone (carbonyl) - high reactivity - atoms (42, 43, 40, 44) - PGs available: yes
  [3] Ester (ester) - low reactivity - atoms (25, 26, 27, 28) - PGs available: no
  [4] Lactone (cyclic ester) (ester) - low reactivity - atoms (25, 26, 27) - PGs available: no

→ Chemist selects which site(s) to protect

STEP 2: Protecting Group Options Per Site
--------------------------------------------------
  Site [0] Secondary Alcohol (atoms (18, 14, 20, 19)):
    [0] TBS - Score: 0.80  |  TBSCl, imidazole, DM